# 01 — BPO SLA Performance: Data Preparation

Sigue de [`01_eda.ipynb`](./01_eda.ipynb). Acá es donde los hallazgos de la fase anterior se convierten en un dataset que de verdad se puede usar para modelar. En resumen, este notebook hace cinco cosas:

1. Revisa con más detalle (agente por agente) el hallazgo de "inconsistencia de granularidad" del EDA, antes de decidir qué hacer con enero.
2. Construye features temporales y features históricas por agente que no dependen de `avg_aht`, resolviendo el data leakage detectado antes.
3. Codifica las variables categóricas.
4. Hace un split train/test cronológico — no aleatorio — porque el objetivo de negocio es predecir el riesgo de incumplimiento hacia adelante en el tiempo, no interpolar dentro del mismo período.
5. Guarda los datasets procesados para `03_modeling.ipynb`.

In [1]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)

RAW_PATH = '../data/call_metrics_dataset.csv'
TRAIN_OUT = '../data/processed_train.csv'
TEST_OUT = '../data/processed_test.csv'

In [2]:
df = pd.read_csv(RAW_PATH, sep=';', decimal=',', thousands='.')
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values(['date', 'agent_id']).reset_index(drop=True)
df.shape

(270, 7)

## 1. Revisando el hallazgo de "granularidad inconsistente"

El EDA había marcado que enero (27 filas, 1/día) y julio (243 filas, 9/día) parecían tener distinta granularidad. Antes de decidir si se excluye enero, conviene revisar qué agente o agentes aparecen en cada período — si el patrón tiene otra explicación, la decisión de preparación cambia por completo.

In [3]:
jan = df[df['date'].dt.month == 1]
jul = df[df['date'].dt.month == 7]

print('Enero — agentes:', sorted(jan['agent_id'].unique()), '| idiomas:', sorted(jan['lang_id'].unique()))
print('Julio — agentes:', sorted(jul['agent_id'].unique()), '| idiomas:', sorted(jul['lang_id'].unique()))

Enero — agentes: [np.int64(3)] | idiomas: [np.int64(1)]
Julio — agentes: [np.int64(1), np.int64(2), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10)] | idiomas: [np.int64(1), np.int64(2)]


Toca corregir el diagnóstico inicial: no es un problema de granularidad. Cada fila sigue siendo una combinación agente/día/producto/idioma perfectamente válida. Lo que pasa es que en enero solo operaba el agente 3 (y solo en idioma 1); recién en julio aparecen los 10 agentes en paralelo. El dato refleja una expansión real de la operación — un piloto de 1 agente que se convierte en equipo completo de 10 — no un error de captura. Me hizo pensar dos veces antes de tirar esas filas.

Por eso se conservan las 270 filas. Ninguna es inválida ni está mal formada, y descartar enero tiraría información histórica real del agente 3. Sí queda una limitación para documentar: el agente 3 es el único con historial previo a julio, así que sus features históricas (sección 2) van a tener más profundidad que las del resto.

## 2. Feature engineering

### 2.1 Features temporales

Todas derivadas de `date` y disponibles de antemano — no hay leakage posible acá, porque se conocen antes de que ocurra la jornada.

In [4]:
df['day_of_week'] = df['date'].dt.day_name()
df['is_weekend'] = df['date'].dt.dayofweek.isin([5, 6]).astype(int)
df['is_rollout_phase'] = (df['date'].dt.month == 7).astype(int)

df[['date', 'day_of_week', 'is_weekend', 'is_rollout_phase']].drop_duplicates().head()

,date,day_of_week,is_weekend,is_rollout_phase
0,2020-01-01,Wednesday,0,0
1,2020-01-02,Thursday,0,0
2,2020-01-03,Friday,0,0
3,2020-01-04,Saturday,1,0
4,2020-01-05,Sunday,1,0


### 2.2 Features históricas por agente — el reemplazo seguro de `avg_aht`

El EDA confirmó que `avg_aht` define casi directamente `std_pass`, así que queda descartada como feature por el leakage que implica. En su lugar se construyen features de desempeño histórico del agente hasta el día anterior, con una ventana expandiente (`shift()` + `expanding()`): capturan si un agente viene incumpliendo o no, sin tocar información del propio día que se quiere predecir.

Se calculan sobre el dataset completo ordenado por fecha. Esto es seguro incluso antes de separar train/test, porque cada fila solo mira su propio pasado real — nunca el futuro, ni el propio ni el de otro agente.

In [5]:
df['agent_prior_pass_rate'] = (
    df.groupby('agent_id')['std_pass']
      .apply(lambda s: s.shift().expanding().mean())
      .reset_index(level=0, drop=True)
)
df['agent_prior_avg_calls'] = (
    df.groupby('agent_id')['calls_handled']
      .apply(lambda s: s.shift().expanding().mean())
      .reset_index(level=0, drop=True)
)

print('Filas sin historial previo (primera aparición del agente):', df['agent_prior_pass_rate'].isnull().sum())
df[['agent_id', 'date', 'std_pass', 'agent_prior_pass_rate', 'agent_prior_avg_calls']].head(12)

Filas sin historial previo (primera aparición del agente): 10


,agent_id,date,std_pass,agent_prior_pass_rate,agent_prior_avg_calls
0,3,2020-01-01,1,NaN,NaN
1,3,2020-01-02,0,1.000000,17.000000
2,3,2020-01-03,1,0.500000,15.500000
3,3,2020-01-04,1,0.666667,15.333333
4,3,2020-01-05,0,0.750000,16.000000
5,3,2020-01-06,1,0.600000,15.000000
6,3,2020-01-07,1,0.666667,14.666667
7,3,2020-01-08,0,0.714286,15.142857
8,3,2020-01-09,1,0.625000,15.000000
9,3,2020-01-10,1,0.666667,14.777778


Para la primera aparición de cada agente (10 filas, una por agente) no hay historial previo que mirar. Se imputa con un prior neutro: 0.5 para la tasa de cumplimiento, que no favorece cumple ni incumple, y la mediana global de `calls_handled` para el volumen. La idea es evitar meter un valor arbitrario que el modelo termine leyendo como señal real.

In [6]:
global_median_calls = df['calls_handled'].median()

df['agent_prior_pass_rate'] = df['agent_prior_pass_rate'].fillna(0.5)
df['agent_prior_avg_calls'] = df['agent_prior_avg_calls'].fillna(global_median_calls)

assert df['agent_prior_pass_rate'].isnull().sum() == 0
assert df['agent_prior_avg_calls'].isnull().sum() == 0
print('Imputación completa. Mediana global de calls_handled usada como fallback:', global_median_calls)

Imputación completa. Mediana global de calls_handled usada como fallback: 20.0


### 2.3 Retirando `avg_aht` del set de modelado

No se descarta del todo — se guarda aparte, porque sirve para diagnóstico y para la alternativa de reformular el problema como regresión que se discutió en el EDA — pero queda excluida explícitamente de las features.

In [7]:
df_diagnostic_aht = df[['agent_id', 'date', 'product_id', 'lang_id', 'avg_aht', 'std_pass']].copy()
df_model = df.drop(columns=['avg_aht'])
print('Columnas disponibles para modelar:', list(df_model.columns))

Columnas disponibles para modelar: ['agent_id', 'date', 'product_id', 'lang_id', 'calls_handled', 'std_pass', 'day_of_week', 'is_weekend', 'is_rollout_phase', 'agent_prior_pass_rate', 'agent_prior_avg_calls']


## 3. Codificación de variables categóricas

`agent_id`, `product_id` y `lang_id` son identificadores nominales pese a ser enteros — un id más alto no implica "más" de nada, así que tratarlos como numéricos sería un error. Se codifican con one-hot (`drop_first=True` para evitar colinealidad perfecta en el baseline lineal; a los modelos de árbol esto no les afecta). Mismo tratamiento para `day_of_week`.

In [8]:
categorical_cols = ['agent_id', 'product_id', 'lang_id', 'day_of_week']

df_encoded = pd.get_dummies(
    df_model,
    columns=categorical_cols,
    prefix=categorical_cols,
    drop_first=True,
)

print('Shape antes:', df_model.shape, '-> después de one-hot:', df_encoded.shape)
df_encoded.head()

Shape antes: (270, 11) -> después de one-hot: (270, 25)


,date,calls_handled,std_pass,is_weekend,is_rollout_phase,agent_prior_pass_rate,agent_prior_avg_calls,agent_id_2,agent_id_3,agent_id_4,agent_id_5,agent_id_6,agent_id_7,agent_id_8,agent_id_9,agent_id_10,product_id_2,product_id_3,lang_id_2,day_of_week_Monday,day_of_week_Saturday,day_of_week_Sunday,day_of_week_Thursday,day_of_week_Tuesday,day_of_week_Wednesday
0,2020-01-01,17,1,0,0,0.500000,20.000000,False,True,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,True
1,2020-01-02,14,0,0,0,1.000000,17.000000,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False
2,2020-01-03,15,1,0,0,0.500000,15.500000,False,True,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False
3,2020-01-04,18,1,1,0,0.666667,15.333333,False,True,False,False,False,False,False,False,False,False,True,False,False,True,False,False,False,False
4,2020-01-05,11,0,1,0,0.750000,16.000000,False,True,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False


## 4. Split train/test cronológico

La separación es por fecha — 80% más antiguo a train, 20% más reciente a test — y no aleatoria a propósito. Un split aleatorio mezclaría días de julio en ambos conjuntos y terminaría sobreestimando el desempeño real del modelo, porque en producción el modelo siempre predice sobre fechas que todavía no pasaron.

In [9]:
unique_dates = sorted(df_encoded['date'].unique())
cutoff_date = unique_dates[int(len(unique_dates) * 0.8)]

train_df = df_encoded[df_encoded['date'] < cutoff_date].copy()
test_df = df_encoded[df_encoded['date'] >= cutoff_date].copy()

print(f'Fecha de corte: {cutoff_date.date()}')
print(f'Train: {len(train_df)} filas ({train_df["date"].min().date()} -> {train_df["date"].max().date()})')
print(f'Test:  {len(test_df)} filas ({test_df["date"].min().date()} -> {test_df["date"].max().date()})')

Fecha de corte: 2020-07-17
Train: 171 filas (2020-01-01 -> 2020-07-16)
Test:  99 filas (2020-07-17 -> 2020-07-27)


In [10]:
print('Balance de clases (std_pass):')
print('  Train:', train_df['std_pass'].mean().round(3), f'({len(train_df)} filas)')
print('  Test: ', test_df['std_pass'].mean().round(3), f'({len(test_df)} filas)')
print()
print('Agentes presentes en train:', sorted(df.loc[train_df.index, 'agent_id'].unique()))
print('Agentes presentes en test: ', sorted(df.loc[test_df.index, 'agent_id'].unique()))

Balance de clases (std_pass):
  Train: 0.585 (171 filas)
  Test:  0.646 (99 filas)

Agentes presentes en train: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10)]
Agentes presentes en test:  [np.int64(1), np.int64(2), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10)]


El balance de clases es parecido entre train (~58% cumple) y test (~65% cumple), así que no hace falta rebalancear nada. El agente 3 no tiene filas en las últimas fechas de julio, entonces no aparece en test — el modelo se evalúa igual sobre el resto de agentes, pero queda anotado como limitación: no hay forma de verificar su desempeño específicamente para el agente 3 en el período de test.

## 5. Guardado de datasets procesados

No quedan versionados en git — excluidos por `.gitignore`, igual que el CSV crudo — porque este notebook los regenera de forma determinística a partir de `call_metrics_dataset.csv`. El escalado de features numéricas para el baseline de regresión logística se deja para `03_modeling.ipynb`, ajustado únicamente sobre train dentro de un `Pipeline` de scikit-learn, para no filtrar estadísticas del test set en el preprocesamiento.

In [11]:
train_df.to_csv(TRAIN_OUT, index=False)
test_df.to_csv(TEST_OUT, index=False)

print('Guardado:', TRAIN_OUT, train_df.shape)
print('Guardado:', TEST_OUT, test_df.shape)

Guardado: ../data/processed_train.csv (171, 25)
Guardado: ../data/processed_test.csv (99, 25)


## 6. Cómo queda el dataset después de esta etapa

1. Enero no se descarta. Lo que el EDA marcó como "inconsistencia de granularidad" resultó ser la expansión del piloto (agente 3) al equipo completo de 10 agentes en julio. Las 270 filas son válidas y se conservan todas.
2. El leakage queda resuelto: `avg_aht` sale del set de modelado y se reemplaza por `agent_prior_pass_rate` y `agent_prior_avg_calls`, desempeño histórico del agente calculado únicamente con información estrictamente anterior a la fila que se predice.
3. `agent_id`, `product_id`, `lang_id` y `day_of_week` quedan codificadas con one-hot (`drop_first=True`).
4. El split es cronológico, no aleatorio: train hasta 2020-07-16, test desde 2020-07-17 en adelante. Así se evalúa al modelo en la tarea real — predecir hacia adelante — y no en un ejercicio de interpolación.
5. Queda una limitación documentada: el agente 3 es el único con historial anterior a julio y no tiene filas en el período de test, así que sus features históricas son más profundas que las del resto, pero su desempeño no se puede validar en el holdout.

Siguiente paso: `03_modeling.ipynb`, donde se entrena el modelo de clasificación de riesgo de incumplimiento de SLA — regresión logística como baseline, XGBoost como modelo principal — sobre `processed_train.csv`, evaluado en `processed_test.csv`.